In [ ]:
import pandas as pd
import asyncio
import aiohttp
from pathlib import Path
from tqdm.asyncio import tqdm
import re 

In [2]:
DATA_DIR = Path.cwd().parent / "data"
ITEMS_PATH = DATA_DIR / "items.csv"
OUTPUT_PATH = DATA_DIR / "enriched_items.csv"

In [4]:
async def fetch_book_data(session, item_id, isbn, semaphore):
    """
    Fetches book metadata from OpenLibrary using the ISBN.
    Uses a semaphore to prevent overwhelming the API.
    """
    if pd.isna(isbn) or str(isbn).strip() == "":
        return {"item_id": item_id, "description": "", "pages": None, "api_found": False}

    # OpenLibrary API endpoint using ISBN
    url = f"https://openlibrary.org/api/books?bibkeys=ISBN:{isbn}&format=json&jscmd=data"
    
    async with semaphore:
        try:
            async with session.get(url, timeout=10) as response:
                if response.status == 200:
                    data = await response.json()
                    book_key = f"ISBN:{isbn}"
                    
                    if book_key in data:
                        book_info = data[book_key]
                        
                        # OpenLibrary sometimes stores descriptions as text, sometimes it's missing from this endpoint.
                        # We grab whatever useful text context we can (e.g., subjects, notes).
                        subjects = [s['name'] for s in book_info.get('subjects', [])]
                        subject_text = ", ".join(subjects)
                        
                        return {
                            "item_id": item_id,
                            "description": book_info.get('notes', subject_text), # Fallback to subjects if no notes
                            "pages": book_info.get('number_of_pages', None),
                            "api_found": True
                        }
                
                # Small sleep to respect rate limits
                await asyncio.sleep(0.1)
                
        except Exception as e:
            # Catch timeouts or connection errors silently to keep the loop running
            pass
            
    return {"item_id": item_id, "description": "", "pages": None, "api_found": False}

async def main():
    # 1. Load the original items
    print("Loading items.csv")
    items_df = pd.read_csv(ITEMS_PATH)
    
    # We will limit concurrency to 10 to avoid getting temporarily banned by OpenLibrary
    semaphore = asyncio.Semaphore(10)
    
    # 2. Set up the async session and tasks
    async with aiohttp.ClientSession() as session:
        tasks = []
        for _, row in items_df.iterrows():
            # Adjust column names based on your actual items.csv header (e.g., 'item_id', 'ISBN')
            task = fetch_book_data(session, row['i'], row['ISBN Valid'], semaphore) 
            tasks.append(task)
        
        print(f"Fetching metadata for {len(tasks)} books from OpenLibrary")
        
        # 3. Run tasks with a progress bar
        results = await tqdm.gather(*tasks)
    
    # 4. Merge results back into the original dataframe
    print("Processing results")
    enriched_data = pd.DataFrame(results)
    
    # Merge on the item ID (assuming your items.csv uses 'i' like the interactions df)
    final_df = items_df.merge(enriched_data, left_on='i', right_on='item_id', how='left')
    final_df.drop(columns=['item_id'], inplace=True)
    
    # 5. Save to CSV
    final_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Success! Enriched data saved to {OUTPUT_PATH}")
    
    # Print a quick summary of how many books we successfully found
    found_count = final_df['api_found'].sum()
    print(f"Found additional data for {found_count} out of {len(final_df)} books.")

# Run the async loop
await main()

Loading items.csv
Fetching metadata for 15291 books from OpenLibrary


100%|██████████| 15291/15291 [1:20:53<00:00,  3.15it/s]


Processing results
Success! Enriched data saved to c:\Users\Julien\OneDrive\VS Code\OneDrive\GitHub\library-recommender\data\enriched_items.csv
Found additional data for 97 out of 15291 books.


In [ ]:
import re 
# Set up paths 
DATA_DIR = Path.cwd().parent / "data"
ITEMS_PATH = DATA_DIR / "items.csv"
OUTPUT_PATH = DATA_DIR / "enriched_items_v2.csv"

def extract_valid_isbns(raw_isbn_str):
    """
    Splits by semicolon, removes hyphens/spaces, and filters for 10 or 13 length codes.
    """
    if pd.isna(raw_isbn_str) or str(raw_isbn_str).strip() == "":
        return []
    
    raw_codes = str(raw_isbn_str).split(';')
    valid_isbns = []
    
    for code in raw_codes:
        # Strip out dashes and whitespace
        clean_code = re.sub(r'[\s-]', '', code)
        
        # A basic heuristic: ISBNs are generally 10 or 13 characters long.
        # (ISBN-10 can sometimes end in an 'X')
        if len(clean_code) in [10, 13]:
            valid_isbns.append(clean_code)
            
    return valid_isbns

async def fetch_book_data(session, item_id, isbns, semaphore):
    """
    Queries OpenLibrary using a list of multiple ISBNs.
    """
    # If no valid ISBNs were extracted, skip the API call
    if not isbns:
        return {"item_id": item_id, "description": "", "pages": None, "api_found": False}

    # OpenLibrary allows batching keys! Example: "ISBN:123,ISBN:456"
    bibkeys = ",".join([f"ISBN:{isbn}" for isbn in isbns])
    url = f"https://openlibrary.org/api/books?bibkeys={bibkeys}&format=json&jscmd=data"
    
    async with semaphore:
        try:
            async with session.get(url, timeout=10) as response:
                if response.status == 200:
                    data = await response.json()
                    
                    # Check which of the ISBNs returned a hit
                    for isbn in isbns:
                        book_key = f"ISBN:{isbn}"
                        if book_key in data:
                            book_info = data[book_key]
                            
                            subjects = [s['name'] for s in book_info.get('subjects', [])]
                            subject_text = ", ".join(subjects)
                            
                            return {
                                "item_id": item_id,
                                "description": book_info.get('notes', subject_text), 
                                "pages": book_info.get('number_of_pages', None),
                                "api_found": True
                            }
                
                # Small sleep to respect rate limits
                await asyncio.sleep(0.1)
                
        except Exception as e:
            pass
            
    return {"item_id": item_id, "description": "", "pages": None, "api_found": False}

async def main():
    print("Loading items.csv...")
    items_df = pd.read_csv(ITEMS_PATH)
    
    semaphore = asyncio.Semaphore(10)
    
    async with aiohttp.ClientSession() as session:
        tasks = []
        for _, row in items_df.iterrows():
            # 1. Parse the ISBN column first
            isbns_list = extract_valid_isbns(row['isbn'])
            
            # 2. Pass the list to our API fetcher
            task = fetch_book_data(session, row['i'], isbns_list, semaphore) 
            tasks.append(task)
        
        print(f"Fetching metadata for {len(tasks)} books from OpenLibrary...")
        results = await tqdm.gather(*tasks)
    
    print("Processing results...")
    enriched_data = pd.DataFrame(results)
    
    final_df = items_df.merge(enriched_data, left_on='i', right_on='item_id', how='left')
    final_df.drop(columns=['item_id'], inplace=True)
    
    final_df.to_csv(OUTPUT_PATH, index=False)
    
    found_count = final_df['api_found'].sum()
    print(f"Success! Enriched data saved to {OUTPUT_PATH}")
    print(f"Found additional data for {found_count} out of {len(final_df)} books.")

await main()